In this tutorial/guide, we will be exploring the Agents and Workflows (this content is derived from Anthropic skilljar site)

Workflows and agents are strategies for handling user tasks that can't be completed by Claude in a single request.

We should use workflows when:

1. We can picture that Claude should go through to solve a problem, or when the app's UX constrains users to a set of tasks.

2. When you're not sure exactly what task or task parameters you'll give to Claude



Imagine if we have a web app where users drag and drop an image of a metal part, and you create a STEP file from it. Since we have a pretty good idea of exactly what to do when a user supplies an image file, and we can easily write all of this out with code as a predefined series of steps.

Here's how an Image to CAD workflow would look like.

1. Feed an image into Claude, asking it to describe the object
2. Based on the description, ask Claude to use the CadQuery library to model the object
3. Create a rendering
4. Ask Claude to grade the rendering against the original image. If there are issues, fix them

This modeling workflow is an example of an evaluator-optimizer pattern. Here's how it works:

- Producer: Takes input and creates output (Claude using CadQuery to model the part and create a rendering)
- Grader: Evaluates the output against some criteria
- Feedback loop: If the grader doesn't accept the output, feedback goes back to the producer for improvement
- Iteration: The cycle repeats until the grader accepts the output




Parallelization workflows can help break down complex tasks into manageable and focused pieces.

Instead of cramming everything into one request (and overload Claude), we can split the task into multiple parallel requests. Each request will focus on evaluating the part for a single material type with specialized criteria.
Here's an example of workflow to demostrate how it works:

1. Send the same image to Claude multiple times simultaneously
2. Each request includes specialized criteria for one material (lets say our application is a material designer)
3. Claude evaluates the part's suitability for each material independently
4. Collect all the analysis results and feed them into a final aggregation step

The final step then sends all the individual analysis results back to Claude with a request to compare them and make a final material recommendation.

Overall, the parallelization pattern follows a simple structure:

- Split a single task into multiple sub-tasks
- Run the subtasks in parallel
- Aggregate the results together
- The parallelized sub-tasks don't need to be identical (so this isn't similar to algorithms such as Divide and Conquer or Dynamic Programming :))

Chaining workflow is another approach when dealing with complex tasks or long prompts that Claude struggles to handle consistently.

A chaining workflow breaks down a large, complex task into smaller, sequential subtasks.
Instead of asking Claude to do everything at once, we split the work into focused steps that build on each other.

The chaining approach offers several advantages:
- Split large tasks into smaller, but non-parallelizable subtasks
- Optionally do non-LLM processing between each task
- Keep Claude focused on one aspect of the overall task

Routing workflows is another way to solve a common problem in AI applications.
The routing process happens in 2 steps:

1. Categorization - Send the user's topic to Claude with a request to categorize it into one of your predefined genres

2. Specialized Processing - Use the category result to select the appropriate prompt template and generate content



A routing workflow follows this pattern:

- User input goes to a router component first
- The router categorizes the request using an initial Claude call
- Based on the category, the input gets forwarded to one specific processing pipeline
- Each pipeline can have its own workflow, prompts, or tools optimized for that category

Meanwhile, Agents represent a shift from the structured workflows. The real power of agents lies in their ability to combine simple tools in unexpected ways.

For example we have the following basic set of datetime tools:

- `get_current_datetime`
- `add_duration_to_datetime`
- `set_reminder`

These tools seem simple individually, but Claude can chain them together to handle complex requests.

For "What's the time?" Claude simple calls `get_current_datetime`
For "What day of the week is it in 11 days", it chains `get_current_datetime` followed by `add_duration_to_datetime`


The important thing for building effective agents is providing reasonably abstract tools rather than hyper-specialized ones.
For example, Claude Code (a coding agent) has access to generic, flexible tools like:

- `bash` -> run any command
- `read` -> read any file
- `write` -> create any file
- `edit` -> modify files
- `glob` -> find files
- `grep` -> search file contents

Note that it doesn't have specialized tools like "refactor code" or "install dependencies".
Instead, Claude figures out how to use the basic tools to accomplish these complex tasks. This abstraction allows it to handle countless programming scenarios that the developers never explicitly planned for.



When designing agents, provide tools that Claude can combine in creative ways.

For example, a social media video agent might include:
- `bash` -> access to FFMPEG for video processing
- `generate_image`-> Create images from prompts
- `text_to_speech` -> Convert text to audio
- `post_media` -> Upload content to social platforms

This tool set enables both simple workflows (create and post a video) and more interactive experiences where the agent might generate a sample image first, get user approval, then proceed with video creation.

The agent can adapt its approach based on user feedback and preferences, something that is difficult to achieve with a rigid workflow.

When building AI agents, there is one crucial concept that gets overlooked -> environment inspection.

Why does it matter? Think about how Claude works with computer use. Every time Claude performs an action like typing text or clicking a button, it immediately receives a screenshot to understand what happened. From Claude's perspective, clicking a button could navigate to a new page, open a menu, or trigger any number of changes. Without being able to see the results, Claude has no way to understand whether its action succeeded or what the new state of the environment looks like.

The same thing applies to file operations. Before Claude can modify any file, it needs to understand the current contents.

You can guide Claude to inspect its environment through system prompts.
For example, for a video creation agent that needs to:
- Generate video content using tools like FFmpeg
- Verify that audio dialogue is placed correctly
- Check that visual elements appear as expected

You can include system prompt instructions like:
- Use the bash tool to run whisper.cpp and generate caption files with timestamps to verify dialogue placement
- Use FFmpeg to extract screenshots from the video at regular intervals to visually inspect the output
- Compare the generated content against the original requirements

When Claude can inspect its environment, it can improve:
- Better progress tracking
- Error handling
- Quality assurance
- Adaptive behavior